# PuffeRL Training on TuranPufferEnv
Uses PufferLib 3.0's `PuffeRL` trainer with our vectorised C environment.

In [6]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

import torch
import numpy as np
import matplotlib.pyplot as plt

import pufferlib
import pufferlib.vector
import pufferlib.models
import pufferlib.pytorch
from pufferlib.pufferl import PuffeRL

from turan_puffer_env import TuranPufferEnv
from turan_env_c import CHECKER_C3, CHECKER_C4, CHECKER_K4

print(f"pufferlib {pufferlib.__version__}  torch {torch.__version__}")
print(f"cuda={torch.cuda.is_available()}  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")

pufferlib 3.0  torch 2.10.0+cu130
cuda=True  NVIDIA GeForce RTX 5090


## Setup

In [7]:
# ── CHANGE THESE ──────────────────────────────────────────────────────────────
N          = 20
NUM_ENVS   = 1024        # parallel envs in C (all handled by one TuranPufferEnv)
CHECKER    = CHECKER_C3
TOTAL_STEPS = 20_000_000
# ──────────────────────────────────────────────────────────────────────────────

CKPT_DIR = f'./checkpoints_puffer_n{N}'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"n={N}  envs={NUM_ENVS}  checker={CHECKER}  ckpt={CKPT_DIR}")

n=20  envs=1024  checker=0  ckpt=./checkpoints_puffer_n20


## Build vecenv and policy

In [8]:
# Serial with num_envs=1: our TuranPufferEnv is already fully vectorised in C,
# so one instance is all we need — no Python-level multiprocessing overhead.
vecenv = pufferlib.vector.Serial(
    env_creators = [TuranPufferEnv],
    env_args     = [[]],
    env_kwargs   = [{'n': N, 'num_envs': NUM_ENVS, 'checker_id': CHECKER}],
    num_envs     = 1,
)

# Default MLP policy: flattens obs → Linear → GELU → action logits + value
policy = pufferlib.models.Default(
    env         = vecenv,
    hidden_size = 256,
).to('cuda')

print(f"obs  space : {vecenv.single_observation_space}")
print(f"act  space : {vecenv.single_action_space}")
print(f"num_agents : {vecenv.num_agents}")
print(f"params     : {sum(p.numel() for p in policy.parameters()):,}")

obs  space : Box(0, 1, (190,), uint8)
act  space : Discrete(190)
num_agents : 1024
params     : 97,983


## Config

In [9]:
config = dict(
    # rollout
    total_timesteps    = TOTAL_STEPS,
    batch_size         = 'auto',   # = num_agents * bptt_horizon
    bptt_horizon       = 64,       # steps per segment (our episode is ~n*(n-1)/2 steps)
    # PPO
    update_epochs      = 10,
    minibatch_size     = 2048,
    max_minibatch_size = 2048,
    clip_coef          = 0.2,
    vf_clip_coef       = 10.0,     # generous — don't over-constrain value
    vf_coef            = 0.5,
    ent_coef           = 0.01,
    gamma              = 0.99,
    gae_lambda         = 0.95,
    # vtrace (disabled: set both clips very high)
    vtrace_rho_clip    = 1.0,
    vtrace_c_clip      = 1.0,
    # priority replay (disabled)
    prio_alpha         = 0.0,
    prio_beta0         = 1.0,
    # optimiser
    optimizer          = 'adam',
    learning_rate      = 3e-4,
    adam_beta1         = 0.9,
    adam_beta2         = 0.999,
    adam_eps           = 1e-8,
    max_grad_norm      = 0.5,
    anneal_lr          = False,
    # system
    device             = 'cuda',
    precision          = 'bfloat16',
    cpu_offload        = False,
    use_rnn            = False,
    compile            = True,
    compile_mode       = 'default',
    compile_fullgraph  = False,
    torch_deterministic= False,
    seed               = 1,
    # checkpointing / logging
    checkpoint_interval = 100,
    checkpoint_path    = CKPT_DIR,
    env                = f'turan_n{N}_checker{CHECKER}',
    run_name           = f'turan_n{N}',
    run_dir            = CKPT_DIR,
    data_dir           = CKPT_DIR,
)

for k, v in config.items():
    print(f"  {k:25s} = {v}")

  total_timesteps           = 20000000
  batch_size                = auto
  bptt_horizon              = 64
  update_epochs             = 10
  minibatch_size            = 2048
  max_minibatch_size        = 2048
  clip_coef                 = 0.2
  vf_clip_coef              = 10.0
  vf_coef                   = 0.5
  ent_coef                  = 0.01
  gamma                     = 0.99
  gae_lambda                = 0.95
  vtrace_rho_clip           = 1.0
  vtrace_c_clip             = 1.0
  prio_alpha                = 0.0
  prio_beta0                = 1.0
  optimizer                 = adam
  learning_rate             = 0.0003
  adam_beta1                = 0.9
  adam_beta2                = 0.999
  adam_eps                  = 1e-08
  max_grad_norm             = 0.5
  anneal_lr                 = False
  device                    = cuda
  precision                 = bfloat16
  cpu_offload               = False
  use_rnn                   = False
  compile                   = True
  compile_mode   

## Train

In [10]:
trainer = PuffeRL(config=config, vecenv=vecenv, policy=policy)

history = []
while trainer.global_step < config['total_timesteps']:
    trainer.evaluate()   # collect rollout
    logs = trainer.train()  # PPO update
    if logs:
        history.append({
            'step': trainer.global_step,
            'sps':  trainer.sps,
            **{k: v for k, v in logs.items()},
        })

print('Training done.')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  PufferLib 3.0     🐡                       CPU: 0.0%         GPU: 0.0%        DRAM: 0.0%           VRAM: 0.0%  │
│                                                                                                                 │
│  Summary                      Value    Evaluate                 0s     0%    Losses                      Value  │
│  Env             turan_n20_checker0      Forward                0s     0%                                       │
│  Params                       98.0K      Env                    0s     0%                                       │
│  Steps                            0      Copy                   0s     0%                                       │
│  SPS                              0      Misc                   0s     0%                                       │
│  Epoch                            0    Train                    0s     

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3 history = []                                                                                │
│    4 while trainer.global_step < config['total_timesteps']:                                      │
│    5 │   trainer.evaluate()   # collect rollout                                                  │
│ ❱  6 │   logs = trainer.train()  # PPO update                                                    │
│    7 │   if logs:                                                                                │
│    8 │   │   history.append({                                                                    │
│    9 │   │   │   'step': trainer.global_step,                                                    │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/erro │
│ rs/__init__.py:362 in wrapper                                                                    │
│                                                                                                  │
│   359 │   │   │   assert error_handler is not None  # assertion for mypy type checker            │
│   360 │   │   │   error_handler.initialize()                                                     │
│   361 │   │   │   try:                                                                           │
│ ❱ 362 │   │   │   │   return f(*args, **kwargs)                                                  │
│   363 │   │   │   except SystemExit as se:                                                       │
│   364 │   │   │   │   # For run_path based entrypoints, SystemExit with code = 0 will never ex   │
│   365 │   │   │   │   # Handling it here by returning a value:                                   │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/pufferlib/pufferl.py:342 in train              │
│                                                                                                  │
│    339 │   │   │   prio_weights = torch.nan_to_num(adv**a, 0, 0, 0)                              │
│    340 │   │   │   prio_probs = (prio_weights + 1e-6)/(prio_weights.sum() + 1e-6)                │
│    341 │   │   │   idx = torch.multinomial(prio_probs, self.minibatch_segments)                  │
│ ❱  342 │   │   │   mb_prio = (self.segments*prio_probs[idx, None])**-anneal_beta                 │
│    343 │   │   │   mb_obs = self.observations[idx]                                               │
│    344 │   │   │   mb_actions = self.actions[idx]                                                │
│    345 │   │   │   mb_logprobs = self.logprobs[idx]                                              │
│                                                                                                  │
│ /root/backtoRL/.venv/lib/python3.12/site-packages/torch/_tensor.py:40 in wrapped                 │
│                                                                                                  │
│     37 def _handle_torch_function_and_wrap_type_error_to_not_implemented(                        │
│     38 │   f: Callable[Concatenate[_TensorLike, _P], "Tensor"],                                  │
│     39 ) -> Callable[Concatenate[_TensorLike, _P], "Tensor"]:                                    │
│ ❱   40 │   @functools.wraps(f)                                                                   │
│     41 │   def wrapped(self: _TensorLike, *args: _P.args, **kwargs: _P.kwargs) -> "Tensor":      │
│     42 │   │   try:                                        

## Training curves

In [ ]:
if not history:
    print('No history recorded yet.')
else:
    print('Available keys:', list(history[0].keys()))

    steps = [h['step'] / 1e6 for h in history]

    def get(key):
        return [h[key] for h in history if key in h]

    fig, axes = plt.subplots(2, 3, figsize=(15, 7))

    for ax, key, title in [
        (axes[0,0], 'episode_return',  'Episode Return'),
        (axes[0,1], 'policy_loss',     'Policy Loss'),
        (axes[0,2], 'value_loss',      'Value Loss'),
        (axes[1,0], 'entropy',         'Entropy'),
        (axes[1,1], 'sps',             'SPS'),
        (axes[1,2], 'clip_fraction',   'Clip Fraction'),
    ]:
        vals = get(key)
        if vals:
            ax.plot(steps[:len(vals)], vals)
        ax.set_title(title); ax.set_xlabel('steps (M)')

    plt.tight_layout()
    plt.show()